# LeetCode 825 — Friends Of Appropriate Ages bằng Python

Link đề: https://leetcode.com/problems/friends-of-appropriate-ages/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def numFriendRequests(self, ages: List[int]) -> int:
        ...
```

Ghi chú: bài này không cần cấu trúc tự dựng. Mọi lời giải được đối chiếu với một oracle làm đúng từng chữ theo ba điều kiện của đề trên **từng cặp người** (`O(n²)`), kể cả vét cạn mọi dãy tuổi nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trên một mạng xã hội có `n` người với tuổi `ages[i]`; người `x` gửi lời mời kết bạn tới người `y` (`x != y`) trừ khi vi phạm một trong ba điều kiện về tuổi; hãy đếm **tổng số lời mời** (có hướng: `x -> y` và `y -> x` là hai lời mời khác nhau).
2. Nắm các công cụ Python: `collections.Counter` / mảng đếm theo tuổi, `itertools.accumulate` (tổng tiền tố), `bisect_left` / `bisect_right` (tìm kiếm nhị phân trên danh sách đã sắp xếp), hai con trỏ trên mảng đã sắp xếp.
3. Đi từ **thử mọi cặp người** (`O(n²)`) → **đếm theo tuổi** rồi duyệt mọi cặp tuổi (chỉ `120 × 120` cặp) → **tổng tiền tố** (`O(n + 120)`) → **sắp xếp + tìm kiếm nhị phân** (`O(n log n)`) → **sắp xếp + hai con trỏ**.
4. Hiểu mấu chốt: điều kiện thực chất là **`0.5·age[x] + 7 < age[y] <= age[x]`** (điều kiện thứ ba là thừa), nên mỗi người `x` chỉ gửi cho những người có tuổi trong một **khoảng liên tiếp** `[age[x]//2 + 8, age[x]]`; chỉ cần đếm số người trong khoảng đó rồi trừ chính mình.
5. Tránh các lỗi kinh điển: quên trừ chính mình, xử lý sai cặp **cùng tuổi** (phải là `cnt·(cnt-1)` chứ không phải `cnt²`), sai biên `<=` / `<` ở điều kiện `0.5·age + 7`, và dùng `O(n²)` với `n = 2·10^4`.

## 2. Đề bài & ví dụ

Có `n` người trên một mạng xã hội. Cho mảng `ages` với `ages[i]` là tuổi của người thứ `i`.

Người `x` **sẽ không** gửi lời mời kết bạn cho người `y` (`x != y`) nếu **một trong ba điều kiện sau đúng**:

1. `age[y] <= 0.5 * age[x] + 7`
2. `age[y] > age[x]`
3. `age[y] > 100 && age[x] < 100`

Ngược lại, `x` **sẽ gửi** lời mời cho `y`.

Chú ý: nếu `x` gửi cho `y` thì `y` chưa chắc gửi lại cho `x`. Và một người không gửi lời mời cho chính mình. Hãy trả về **tổng số lời mời** được gửi.

**Ví dụ 1**

- Input: `ages = [16,16]`
- Output: `2`
- Giải thích: hai người gửi cho nhau.

**Ví dụ 2**

- Input: `ages = [16,17,18]`
- Output: `2`
- Giải thích: các lời mời là `17 -> 16` và `18 -> 17`.

**Ví dụ 3**

- Input: `ages = [20,30,100,110,120]`
- Output: `3`
- Giải thích: các lời mời là `110 -> 100`, `120 -> 110`, `120 -> 100`.

**Ràng buộc:**

- `n == ages.length`
- `1 <= n <= 2 * 10^4`
- `1 <= ages[i] <= 120`

Chú ý: `n` lên tới `2·10^4` nên thử mọi cặp người (`4·10^8` cặp) là quá chậm; nhưng **tuổi chỉ nằm trong `1..120`**, một chi tiết rất đáng tận dụng.

In [1]:
from typing import List
from collections import Counter
from itertools import accumulate
from bisect import bisect_left, bisect_right
import random
import time

examples = [
    ([16, 16], 2),
    ([16, 17, 18], 2),
    ([20, 30, 100, 110, 120], 3),
]
for ages, expected in examples:
    print(f"ages={ages} -> kỳ vọng {expected}")

ages=[16, 16] -> kỳ vọng 2
ages=[16, 17, 18] -> kỳ vọng 2
ages=[20, 30, 100, 110, 120] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Đọc điều kiện: "sẽ gửi" là phủ định của "không gửi"

Ba điều kiện của đề là các lý do để **không** gửi. Người `x` gửi cho `y` khi **cả ba** điều kiện đều **sai**: `age[y] > 0.5·age[x] + 7` **và** `age[y] <= age[x]` **và** không xảy ra (`age[y] > 100` và `age[x] < 100`). Phủ định của "A hoặc B hoặc C" là "không A và không B và không C" (luật De Morgan).

Hãy viết hàm `can_send(ax, ay)` đúng từng chữ theo đề, và thử vài cặp. Lưu ý số thực `0.5 * ax + 7` có thể có phần `.5` khi `ax` lẻ.

In [2]:
def can_send(ax, ay):
    """x (tuổi ax) có gửi lời mời cho y (tuổi ay) không? Đúng từng chữ theo đề (chưa xét x != y)."""
    if ay <= 0.5 * ax + 7:
        return False
    if ay > ax:
        return False
    if ay > 100 and ax < 100:
        return False
    return True

for ax, ay in [(16, 16), (17, 16), (16, 17), (14, 14), (15, 15), (110, 100), (100, 110), (30, 20)]:
    print(f"x tuổi {ax:3d} -> y tuổi {ay:3d}: {can_send(ax, ay)}   (ngưỡng 0.5*{ax}+7 = {0.5 * ax + 7})")

x tuổi  16 -> y tuổi  16: True   (ngưỡng 0.5*16+7 = 15.0)
x tuổi  17 -> y tuổi  16: True   (ngưỡng 0.5*17+7 = 15.5)
x tuổi  16 -> y tuổi  17: False   (ngưỡng 0.5*16+7 = 15.0)
x tuổi  14 -> y tuổi  14: False   (ngưỡng 0.5*14+7 = 14.0)
x tuổi  15 -> y tuổi  15: True   (ngưỡng 0.5*15+7 = 14.5)
x tuổi 110 -> y tuổi 100: True   (ngưỡng 0.5*110+7 = 62.0)
x tuổi 100 -> y tuổi 110: False   (ngưỡng 0.5*100+7 = 57.0)
x tuổi  30 -> y tuổi  20: False   (ngưỡng 0.5*30+7 = 22.0)


### 3.2. Đếm theo giá trị: `Counter` và mảng đếm

Khi giá trị nằm trong một khoảng nhỏ (ở đây tuổi `1..120`), ta không cần xét từng người mà chỉ cần **đếm số người ở mỗi tuổi**: `cnt[a]` = số người tuổi `a`. `Counter(ages)` làm việc đó cho `dict`; hoặc dùng list `cnt = [0] * 121` và `cnt[a] += 1` (chỉ số tuổi là `0..120`, để tuổi `a` nằm tại chỉ số `a`).

In [3]:
ages = [20, 30, 100, 110, 120, 30]
c = Counter(ages)
print(c, "| c[30] =", c[30], "| c[99] =", c[99])
cnt = [0] * 121
for a in ages:
    cnt[a] += 1
print("cnt[20], cnt[30], cnt[100] =", cnt[20], cnt[30], cnt[100], "| tổng =", sum(cnt))

Counter({30: 2, 20: 1, 100: 1, 110: 1, 120: 1}) | c[30] = 2 | c[99] = 0
cnt[20], cnt[30], cnt[100] = 1 2 1 | tổng = 6


### 3.3. Tổng tiền tố (prefix sum) với `accumulate`

Với `cnt`, đặt `pre[i] = cnt[0] + cnt[1] + ... + cnt[i]` (số người có tuổi **không vượt quá** `i`). Khi đó số người có tuổi trong đoạn `[l, r]` là `pre[r] - pre[l - 1]`, tức một phép trừ `O(1)`. `itertools.accumulate` tính tổng tiền tố gọn gàng.

In [4]:
cnt = [0] * 121
for a in [16, 16, 17, 18, 30]:
    cnt[a] += 1
pre = list(accumulate(cnt))
print("pre[15], pre[16], pre[17], pre[18] =", pre[15], pre[16], pre[17], pre[18])
print("số người có tuổi trong [16, 18]:", pre[18] - pre[15])
print("số người có tuổi trong [17, 17]:", pre[17] - pre[16])

pre[15], pre[16], pre[17], pre[18] = 0 2 3 4
số người có tuổi trong [16, 18]: 4
số người có tuổi trong [17, 17]: 1


### 3.4. `bisect_left` và `bisect_right` trên danh sách đã sắp xếp

Với danh sách `s` **đã sắp xếp tăng**:

- `bisect_left(s, v)`: chỉ số đầu tiên mà `s[i] >= v` (vị trí chèn `v` **trước** các phần tử bằng `v`).
- `bisect_right(s, v)`: chỉ số đầu tiên mà `s[i] > v` (vị trí chèn `v` **sau** các phần tử bằng `v`).

Số phần tử trong `[l, r]` là `bisect_right(s, r) - bisect_left(s, l)`. Mỗi lần gọi `O(log n)`.

In [5]:
s = [16, 16, 17, 18, 30, 30]
print("bisect_left(s, 16) =", bisect_left(s, 16), "| bisect_right(s, 16) =", bisect_right(s, 16))
print("bisect_left(s, 30) =", bisect_left(s, 30), "| bisect_right(s, 30) =", bisect_right(s, 30))
print("bisect_left(s, 20) =", bisect_left(s, 20), "(không có 20 -> vị trí chèn)")
print("số phần tử trong [16, 18]:", bisect_right(s, 18) - bisect_left(s, 16))

bisect_left(s, 16) = 0 | bisect_right(s, 16) = 2
bisect_left(s, 30) = 4 | bisect_right(s, 30) = 6
bisect_left(s, 20) = 4 (không có 20 -> vị trí chèn)
số phần tử trong [16, 18]: 4


### 3.5. Hai con trỏ trên mảng đã sắp xếp (đơn điệu)

Nếu danh sách tăng dần và ta xét lần lượt `a = s[0], s[1], ...` (không giảm), thì các **mốc cắt** như "chỉ số đầu tiên có tuổi `> 0.5a + 7`" cũng **chỉ tiến lên, không bao giờ lùi**. Nhờ vậy mỗi con trỏ đi tối đa `n` bước tổng cộng thay vì tìm kiếm lại từ đầu: đó là kỹ thuật **hai con trỏ**. Ví dụ nhỏ: với dãy tăng, con trỏ `j` tìm đầu tiên phần tử `> a/2` chỉ cần tiến tiếp khi `a` tăng.

In [6]:
s = [2, 3, 5, 8, 13, 21]
j = 0
for a in s:
    while j < len(s) and s[j] <= a / 2:          # j chỉ tiến lên
        j += 1
    print(f"a = {a:2d}: phần tử đầu tiên > a/2 = {a / 2:4.1f} nằm ở chỉ số j = {j} (s[j] = {s[j] if j < len(s) else None})")

a =  2: phần tử đầu tiên > a/2 =  1.0 nằm ở chỉ số j = 0 (s[j] = 2)
a =  3: phần tử đầu tiên > a/2 =  1.5 nằm ở chỉ số j = 0 (s[j] = 2)
a =  5: phần tử đầu tiên > a/2 =  2.5 nằm ở chỉ số j = 1 (s[j] = 3)
a =  8: phần tử đầu tiên > a/2 =  4.0 nằm ở chỉ số j = 2 (s[j] = 5)
a = 13: phần tử đầu tiên > a/2 =  6.5 nằm ở chỉ số j = 3 (s[j] = 8)
a = 21: phần tử đầu tiên > a/2 = 10.5 nằm ở chỉ số j = 4 (s[j] = 13)


### 3.6. Đo thời gian với `time.perf_counter`

Để thấy tận mắt `O(n²)` và `O(n)`, ta đo thời gian. `time.perf_counter()` trả về giây dưới dạng số thực; `timeit_once` lấy hiệu hai lần đo.

In [7]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

res, dt = timeit_once(sum, range(10 ** 6))
print("sum =", res, "| khoảng", round(dt * 1000, 2), "ms")

sum = 499999500000 | khoảng 15.2 ms


## 4. Ý tưởng nền tảng / trực giác

Hãy đơn giản hóa điều kiện. Người `x` gửi cho `y` khi

`0.5·age[x] + 7 < age[y]` **và** `age[y] <= age[x]` **và** không (`age[y] > 100` và `age[x] < 100`).

Điều kiện thứ ba **thừa**: nếu `age[y] > 100` và `age[x] < 100` thì `age[y] > age[x]`, đã bị loại bởi điều kiện thứ hai. Vậy rút gọn thành:

**`0.5·age[x] + 7 < age[y] <= age[x]`**.

Tính tay với `ages = [20, 30, 100, 110, 120]`:

| người gửi `x` | khoảng tuổi nhận | ai trong dãy nằm trong khoảng (khác chính mình) | số lời mời |
|---|---|---|---|
| 20 | `(17, 20]` | chỉ có chính 20 | 0 |
| 30 | `(22, 30]` | chỉ có chính 30 | 0 |
| 100 | `(57, 100]` | chỉ có chính 100 | 0 |
| 110 | `(62, 110]` | 100 (và chính 110) | 1 |
| 120 | `(67, 120]` | 100, 110 (và chính 120) | 2 |

Tổng `3`, khớp ví dụ 3.

Hai nhận xét then chốt:

1. Với mỗi `x`, người nhận là **mọi người có tuổi trong một khoảng liên tiếp**; số lượng là một phép đếm trên đoạn.
2. Chính `x` luôn nằm trong khoảng của mình **khi khoảng không rỗng** (vì `age[x] <= age[x]` và `age[x] > 0.5·age[x] + 7` khi `age[x] > 14`); nên phải **trừ 1** để bỏ chính mình. Với `age[x] <= 14` thì khoảng rỗng (không ai nhận được, kể cả chính mình).

Hãy kiểm chứng điều kiện thứ ba là thừa bằng code, thử mọi cặp tuổi `1..120`.

In [8]:
def can_send_simplified(ax, ay):
    return 0.5 * ax + 7 < ay <= ax

diff = [(ax, ay) for ax in range(1, 121) for ay in range(1, 121) if can_send(ax, ay) != can_send_simplified(ax, ay)]
print("số cặp tuổi (1..120) mà điều kiện đầy đủ và rút gọn khác nhau:", len(diff))
assert not diff

# Mọi người tuổi <= 14 không gửi được cho ai, kể cả cùng tuổi
print("tuổi <= 14 có thể gửi cho ai không?", any(can_send(a, b) for a in range(1, 15) for b in range(1, 121)))
print("tuổi 15 gửi được cho:", [b for b in range(1, 121) if can_send(15, b)], "| tuổi 16:", [b for b in range(1, 121) if can_send(16, b)])

số cặp tuổi (1..120) mà điều kiện đầy đủ và rút gọn khác nhau: 0
tuổi <= 14 có thể gửi cho ai không? False
tuổi 15 gửi được cho: [15] | tuổi 16: [16]


## 5. Cách 1 — Brute Force: thử mọi cặp người

### Ý tưởng

Với mỗi cặp **có thứ tự** `(i, j)` với `i != j`, kiểm tra ba điều kiện của đề; nếu `i` gửi được cho `j` thì cộng `1`.

### Code

In [9]:
def count_brute(ages):
    n = len(ages)
    ans = 0
    for i in range(n):
        for j in range(n):
            if i != j and can_send(ages[i], ages[j]):
                ans += 1
    return ans

for ages, expected in examples:
    got = count_brute(ages)
    print(ages, "->", got)
    assert got == expected

[16, 16] -> 2
[16, 17, 18] -> 2
[20, 30, 100, 110, 120] -> 3


### Độ phức tạp và nút thắt

- **Time:** `n²` cặp có thứ tự, mỗi cặp kiểm tra `O(1)`: `O(n²)`. Với `n = 2·10^4` là `4·10^8`.
- **Space:** `O(1)`.

Nút thắt: ta xét từng **người** trong khi chỉ có `120` giá trị tuổi khác nhau; rất nhiều cặp lặp lại cùng một cặp tuổi. Hãy đo để thấy sự bùng nổ.

In [10]:
rng = random.Random(1)
for n in (250, 500, 1000, 2000):
    ages = [rng.randint(1, 120) for _ in range(n)]
    res, dt = timeit_once(count_brute, ages)
    print(f"n = {n:5d} -> đáp án {res:8d}, thời gian {dt * 1000:8.1f} ms")
print("gấp đôi n -> thời gian gấp ~4 lần: n = 2*10^4 sẽ lâu gấp ~100 lần n = 2000")

n =   250 -> đáp án    13672, thời gian      7.6 ms
n =   500 -> đáp án    46375, thời gian     53.9 ms


n =  1000 -> đáp án   206213, thời gian    142.5 ms


n =  2000 -> đáp án   802811, thời gian    545.5 ms
gấp đôi n -> thời gian gấp ~4 lần: n = 2*10^4 sẽ lâu gấp ~100 lần n = 2000


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1 (đếm theo tuổi):** người cùng tuổi hành xử **giống hệt nhau**. Thay vì xét `n` người, xét `120` loại tuổi với `cnt[a]` là số người tuổi `a`. Nếu tuổi `a` gửi được cho tuổi `b` thì **mọi** người tuổi `a` gửi cho **mọi** người tuổi `b`: đóng góp `cnt[a] · cnt[b]` lời mời. Riêng khi `a == b`, một người **không** gửi cho chính mình, nên chỉ có `cnt[a] · (cnt[a] - 1)`.

Số cặp tuổi chỉ `120² = 14400`, không phụ thuộc `n`!

**Bước đột phá 2 (khoảng liên tiếp + tổng tiền tố):** với tuổi `a`, các tuổi `b` hợp lệ là `0.5a + 7 < b <= a`. Vì `b` nguyên: `b >= a//2 + 8` (kiểm tra: `a` chẵn thì `0.5a + 7 = a/2 + 7`, `b >= a/2 + 8`; `a` lẻ thì `0.5a + 7 = (a-1)/2 + 7.5`, `b >= (a-1)/2 + 8 = a//2 + 8`). Vậy `b ∈ [a//2 + 8, a]`, và số người trong khoảng là `pre[a] - pre[a//2 + 7]` (một phép trừ tiền tố). Trừ đi chính mình.

Thử bằng số: kiểm chứng biểu thức `a//2 + 8` cho mọi tuổi và cho cả hai tính chẵn lẻ.

In [11]:
for a in range(1, 121):
    valid_b = [b for b in range(1, 121) if can_send(a, b)]
    if a >= 15:
        assert valid_b == list(range(a // 2 + 8, a + 1)), a
    else:
        assert valid_b == [], a
for a in (15, 16, 17, 20, 21, 100, 101, 120):
    print(f"a = {a:3d}: 0.5a+7 = {0.5 * a + 7:5.1f} -> b từ {a // 2 + 8:3d} đến {a:3d} ({a - (a // 2 + 8) + 1} tuổi)")
print("với a <= 14 khoảng rỗng, với a >= 15 luôn là [a//2 + 8, a]")

a =  15: 0.5a+7 =  14.5 -> b từ  15 đến  15 (1 tuổi)
a =  16: 0.5a+7 =  15.0 -> b từ  16 đến  16 (1 tuổi)
a =  17: 0.5a+7 =  15.5 -> b từ  16 đến  17 (2 tuổi)
a =  20: 0.5a+7 =  17.0 -> b từ  18 đến  20 (3 tuổi)
a =  21: 0.5a+7 =  17.5 -> b từ  18 đến  21 (4 tuổi)
a = 100: 0.5a+7 =  57.0 -> b từ  58 đến 100 (43 tuổi)
a = 101: 0.5a+7 =  57.5 -> b từ  58 đến 101 (44 tuổi)
a = 120: 0.5a+7 =  67.0 -> b từ  68 đến 120 (53 tuổi)
với a <= 14 khoảng rỗng, với a >= 15 luôn là [a//2 + 8, a]


## 7. Các cách tiếp theo

Tổng cộng 5 cách:

1. Thử mọi cặp người (đã làm).
2. **Đếm theo tuổi rồi duyệt mọi cặp tuổi** (`120 × 120`).
3. **Tổng tiền tố** trên mảng đếm (`O(n + 120)`).
4. **Sắp xếp + tìm kiếm nhị phân** (`O(n log n)`), không cần giới hạn tuổi nhỏ.
5. **Sắp xếp + hai con trỏ** (sau khi sắp xếp là `O(n)`).

### 7.1. Cách 2 — Đếm theo tuổi, duyệt mọi cặp tuổi

Giải thích từng đoạn:

1. `cnt[a]` = số người tuổi `a` (mảng 121 phần tử).
2. Với mỗi cặp tuổi `(a, b)` mà `cnt[a]` và `cnt[b]` đều dương và `can_send(a, b)`:
   - cộng `cnt[a] * cnt[b]` (mọi người tuổi `a` gửi cho mọi người tuổi `b`);
   - nếu `a == b`, trừ `cnt[a]` (bỏ cặp "gửi cho chính mình").

In [12]:
def count_by_age_pairs(ages):
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    ans = 0
    for a in range(1, 121):
        if cnt[a] == 0:
            continue
        for b in range(1, 121):
            if cnt[b] == 0:
                continue
            if can_send(a, b):
                ans += cnt[a] * cnt[b]
                if a == b:
                    ans -= cnt[a]
    return ans

for ages, expected in examples:
    got = count_by_age_pairs(ages)
    print(ages, "->", got)
    assert got == expected

[16, 16] -> 2
[16, 17, 18] -> 2
[20, 30, 100, 110, 120] -> 3


In [13]:
class Solution2:
    def numFriendRequests(self, ages: List[int]) -> int:
        cnt = [0] * 121
        for a in ages:
            cnt[a] += 1
        ans = 0
        for a in range(1, 121):
            for b in range(1, a + 1):
                if cnt[a] and cnt[b] and b > 0.5 * a + 7:
                    ans += cnt[a] * cnt[b]
                    if a == b:
                        ans -= cnt[a]
        return ans

for ages, expected in examples:
    assert Solution2().numFriendRequests(ages) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp Cách 2:** Time `O(n + 120²)` (đếm `O(n)` rồi `14400` cặp tuổi); Space `O(121)`. Với ràng buộc này đây đã rất nhanh, nhưng nó **phụ thuộc** vào việc tuổi bị chặn bởi `120`.

### 7.2. Cách 3 — Tổng tiền tố

Giải thích từng đoạn:

1. `cnt[a]` như trên, `pre = accumulate(cnt)` nên `pre[i]` = số người tuổi `<= i`.
2. Với mỗi tuổi `a` từ `15` đến `120` (tuổi `<= 14` không gửi được): số người nhận (cả chính mình) là `pre[a] - pre[a//2 + 7]` (các tuổi trong `[a//2 + 8, a]`).
3. Mỗi người tuổi `a` gửi cho `pre[a] - pre[a//2 + 7] - 1` người (trừ chính mình). Có `cnt[a]` người tuổi `a`, nên đóng góp `cnt[a] * (pre[a] - pre[a//2 + 7] - 1)`.

In [14]:
def count_prefix(ages):
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    pre = list(accumulate(cnt))
    ans = 0
    for a in range(15, 121):
        if cnt[a]:
            ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)
    return ans

for ages, expected in examples:
    got = count_prefix(ages)
    print(ages, "->", got)
    assert got == expected

[16, 16] -> 2
[16, 17, 18] -> 2
[20, 30, 100, 110, 120] -> 3


In [15]:
class Solution3:
    def numFriendRequests(self, ages: List[int]) -> int:
        cnt = [0] * 121
        for a in ages:
            cnt[a] += 1
        pre = list(accumulate(cnt))
        ans = 0
        for a in range(15, 121):
            if cnt[a]:
                ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)
        return ans

for ages, expected in examples:
    assert Solution3().numFriendRequests(ages) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n + 120)`; Space `O(121)`. Đây là cách ngắn và nhanh nhất cho ràng buộc `ages <= 120`.

### 7.3. Cách 4 — Sắp xếp + tìm kiếm nhị phân

Cách này **không dùng** giới hạn `120` (tuổi có thể lớn tới `10^9`), chỉ cần so sánh được.

Giải thích từng đoạn:

1. Sắp xếp `s = sorted(ages)`.
2. Với mỗi người có tuổi `a`: các tuổi hợp lệ nằm trong `[a//2 + 8, a]`. Chỉ số bắt đầu `lo = bisect_left(s, a//2 + 8)`, chỉ số kết thúc (không gồm) `hi = bisect_right(s, a)`.
3. Nếu `hi > lo` (khoảng không rỗng) thì chính người đó thuộc `[lo, hi)`, nên số người nhận là `hi - lo - 1`.

In [16]:
def count_bisect(ages):
    s = sorted(ages)
    ans = 0
    for a in s:
        lo = bisect_left(s, a // 2 + 8)
        hi = bisect_right(s, a)
        if hi > lo:
            ans += hi - lo - 1
    return ans

for ages, expected in examples:
    got = count_bisect(ages)
    print(ages, "->", got)
    assert got == expected

[16, 16] -> 2
[16, 17, 18] -> 2
[20, 30, 100, 110, 120] -> 3


In [17]:
class Solution4:
    def numFriendRequests(self, ages: List[int]) -> int:
        s = sorted(ages)
        ans = 0
        for a in s:
            lo = bisect_left(s, a // 2 + 8)
            hi = bisect_right(s, a)
            if hi > lo:
                ans += hi - lo - 1
        return ans

for ages, expected in examples:
    assert Solution4().numFriendRequests(ages) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp Cách 4:** Time `O(n log n)` (sắp xếp và `2n` lần tìm kiếm nhị phân); Space `O(n)` cho bản sao đã sắp xếp.

### 7.4. Cách 5 — Sắp xếp + hai con trỏ

Vì `s` tăng dần và ta xét `a = s[0], s[1], ...`, các mốc `lo` (đầu tiên có tuổi `> 0.5a + 7`) và `hi` (đầu tiên có tuổi `> a`) **chỉ tiến lên**. Thay vì tìm kiếm nhị phân, ta dùng hai con trỏ:

- `lo` tiến khi `s[lo] <= 0.5a + 7`, tức `2·s[lo] <= a + 14` (dùng số nguyên, tránh số thực).
- `hi` tiến khi `s[hi] <= a`.
- Nếu `hi > lo` thì cộng `hi - lo - 1`.

In [18]:
def count_two_pointers(ages):
    s = sorted(ages)
    n = len(s)
    lo = hi = 0
    ans = 0
    for a in s:
        while lo < n and 2 * s[lo] <= a + 14:       # s[lo] <= 0.5a + 7: chưa đủ tuổi để nhận
            lo += 1
        while hi < n and s[hi] <= a:                 # cùng tuổi hoặc trẻ hơn: còn trong khoảng
            hi += 1
        if hi > lo:
            ans += hi - lo - 1
    return ans

for ages, expected in examples:
    got = count_two_pointers(ages)
    print(ages, "->", got)
    assert got == expected

[16, 16] -> 2
[16, 17, 18] -> 2
[20, 30, 100, 110, 120] -> 3


In [19]:
class Solution5:
    def numFriendRequests(self, ages: List[int]) -> int:
        s = sorted(ages)
        n = len(s)
        lo = hi = 0
        ans = 0
        for a in s:
            while lo < n and 2 * s[lo] <= a + 14:
                lo += 1
            while hi < n and s[hi] <= a:
                hi += 1
            if hi > lo:
                ans += hi - lo - 1
        return ans

for ages, expected in examples:
    assert Solution5().numFriendRequests(ages) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp Cách 5:** Time `O(n log n)` cho sắp xếp, phần quét hai con trỏ `O(n)`; Space `O(n)`. Nếu dữ liệu đã được sắp xếp thì chỉ `O(n)`.

Bảng so sánh năm cách ở mục 16.

## 8. TRACE từng bước

Trace Cách 3 (tổng tiền tố) trên ví dụ 3: `ages = [20, 30, 100, 110, 120]`.

Với mỗi tuổi `a` có người: khoảng nhận `[a//2 + 8, a]`, số người trong khoảng `pre[a] - pre[a//2 + 7]`, trừ chính mình, nhân `cnt[a]`.

| tuổi `a` | `cnt[a]` | khoảng `[a//2 + 8, a]` | người trong khoảng | trừ chính mình | đóng góp |
|---|---|---|---|---|---|
| 20 | 1 | `[18, 20]` | 1 (20) | 0 | `1·0 = 0` |
| 30 | 1 | `[23, 30]` | 1 (30) | 0 | 0 |
| 100 | 1 | `[58, 100]` | 1 (100) | 0 | 0 |
| 110 | 1 | `[63, 110]` | 2 (100, 110) | 1 | `1·1 = 1` |
| 120 | 1 | `[68, 120]` | 3 (100, 110, 120) | 2 | `1·2 = 2` |

Tổng `0 + 0 + 0 + 1 + 2 = 3`. Hàm dưới in đúng các bước.

In [20]:
def trace_prefix(ages):
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    pre = list(accumulate(cnt))
    ans = 0
    print("ages =", ages)
    for a in range(1, 121):
        if not cnt[a]:
            continue
        if a < 15:
            print(f"tuổi {a:3d} (x{cnt[a]}): a <= 14 -> khoảng rỗng, đóng góp 0")
            continue
        lo = a // 2 + 8
        inside = pre[a] - pre[a // 2 + 7]
        add = cnt[a] * (inside - 1)
        ans += add
        print(f"tuổi {a:3d} (x{cnt[a]}): khoảng [{lo}, {a}] | pre[{a}]={pre[a]}, pre[{a // 2 + 7}]={pre[a // 2 + 7]} -> {inside} người | trừ chính mình -> {inside - 1} | đóng góp {cnt[a]}*{inside - 1} = {add}")
    print("tổng:", ans)
    return ans

assert trace_prefix([20, 30, 100, 110, 120]) == 3

ages = [20, 30, 100, 110, 120]
tuổi  20 (x1): khoảng [18, 20] | pre[20]=1, pre[17]=0 -> 1 người | trừ chính mình -> 0 | đóng góp 1*0 = 0
tuổi  30 (x1): khoảng [23, 30] | pre[30]=2, pre[22]=1 -> 1 người | trừ chính mình -> 0 | đóng góp 1*0 = 0
tuổi 100 (x1): khoảng [58, 100] | pre[100]=3, pre[57]=2 -> 1 người | trừ chính mình -> 0 | đóng góp 1*0 = 0
tuổi 110 (x1): khoảng [63, 110] | pre[110]=4, pre[62]=2 -> 2 người | trừ chính mình -> 1 | đóng góp 1*1 = 1
tuổi 120 (x1): khoảng [68, 120] | pre[120]=5, pre[67]=2 -> 3 người | trừ chính mình -> 2 | đóng góp 1*2 = 2
tổng: 3


Trace ví dụ có nhiều người **cùng tuổi** để thấy vì sao phải trừ chính mình: `ages = [16, 16, 16, 17]`. Ba người tuổi `16`: mỗi người gửi cho hai người cùng tuổi còn lại: `3·2 = 6`. Người tuổi `17` có khoảng `[16, 17]` gồm `3 + 1 = 4` người, trừ chính mình còn `3`: `1·3 = 3`. Tổng `9`.

In [21]:
assert trace_prefix([16, 16, 16, 17]) == 9
assert count_brute([16, 16, 16, 17]) == 9

ages = [16, 16, 16, 17]
tuổi  16 (x3): khoảng [16, 16] | pre[16]=3, pre[15]=0 -> 3 người | trừ chính mình -> 2 | đóng góp 3*2 = 6
tuổi  17 (x1): khoảng [16, 17] | pre[17]=4, pre[15]=0 -> 4 người | trừ chính mình -> 3 | đóng góp 1*3 = 3
tổng: 9


Trace Cách 5 (hai con trỏ) trên ví dụ 2: `ages = [16, 17, 18]`.

| bước | `a` | `lo` sau khi tiến | `hi` sau khi tiến | `hi - lo - 1` | tổng |
|---|---|---|---|---|---|
| 1 | 16 | 0 (`2·16 = 32 > 30`: không tiến) | 1 (`s[0] = 16 <= 16`; `s[1] = 17 > 16`) | `1 - 0 - 1 = 0` | 0 |
| 2 | 17 | 0 (`2·16 = 32 <= 17 + 14 = 31`? không) | 2 | `2 - 0 - 1 = 1` | 1 |
| 3 | 18 | 0 (`32 <= 32`: tiến tới 1; `2·17 = 34 > 32`: dừng) | 3 | `3 - 1 - 1 = 1` | 2 |

Kết quả `2`. Hàm dưới in từng bước.

In [22]:
def trace_two_pointers(ages):
    s = sorted(ages)
    n = len(s)
    lo = hi = 0
    ans = 0
    print("đã sắp xếp:", s)
    for i, a in enumerate(s):
        while lo < n and 2 * s[lo] <= a + 14:
            lo += 1
        while hi < n and s[hi] <= a:
            hi += 1
        add = hi - lo - 1 if hi > lo else 0
        ans += add
        print(f"i={i}, a={a}: lo={lo} hi={hi} -> khoảng chỉ số [{lo}, {hi}) = {s[lo:hi]} -> + {add} (tổng {ans})")
    return ans

assert trace_two_pointers([16, 17, 18]) == 2
print()
assert trace_two_pointers([20, 30, 100, 110, 120]) == 3

đã sắp xếp: [16, 17, 18]
i=0, a=16: lo=0 hi=1 -> khoảng chỉ số [0, 1) = [16] -> + 0 (tổng 0)
i=1, a=17: lo=0 hi=2 -> khoảng chỉ số [0, 2) = [16, 17] -> + 1 (tổng 1)
i=2, a=18: lo=1 hi=3 -> khoảng chỉ số [1, 3) = [17, 18] -> + 1 (tổng 2)

đã sắp xếp: [20, 30, 100, 110, 120]
i=0, a=20: lo=0 hi=1 -> khoảng chỉ số [0, 1) = [20] -> + 0 (tổng 0)
i=1, a=30: lo=1 hi=2 -> khoảng chỉ số [1, 2) = [30] -> + 0 (tổng 0)
i=2, a=100: lo=2 hi=3 -> khoảng chỉ số [2, 3) = [100] -> + 0 (tổng 0)
i=3, a=110: lo=2 hi=4 -> khoảng chỉ số [2, 4) = [100, 110] -> + 1 (tổng 1)
i=4, a=120: lo=2 hi=5 -> khoảng chỉ số [2, 5) = [100, 110, 120] -> + 2 (tổng 3)


## 9. Code nộp LeetCode

Ta chọn **Cách 3 (tổng tiền tố)**: ngắn, `O(n + 120)`, và khai thác đúng ràng buộc `ages <= 120`.

In [23]:
class Solution:
    def numFriendRequests(self, ages: List[int]) -> int:
        cnt = [0] * 121
        for a in ages:
            cnt[a] += 1
        pre = list(accumulate(cnt))
        ans = 0
        for a in range(15, 121):
            if cnt[a]:
                ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)
        return ans


sol = Solution()
for ages, expected in examples:
    got = sol.numFriendRequests(ages)
    print(ages, "->", got)
    assert got == expected

[16, 16] -> 2
[16, 17, 18] -> 2
[20, 30, 100, 110, 120] -> 3


## 10. Vì sao đúng

**Khẳng định 1 (rút gọn điều kiện).** Ba điều kiện "không gửi": (1) `age[y] <= 0.5·age[x] + 7`, (2) `age[y] > age[x]`, (3) `age[y] > 100` và `age[x] < 100`. Nếu (3) đúng thì `age[y] > 100 > age[x]`, nên (2) cũng đúng. Do đó (3) không bao giờ là điều kiện duy nhất loại bỏ cặp nào; "gửi" ⇔ (1) sai và (2) sai ⇔ `0.5·age[x] + 7 < age[y] <= age[x]`. (Mục 4 đã kiểm chứng bằng code trên mọi cặp tuổi `1..120`.)

**Khẳng định 2 (khoảng nguyên).** Với `a = age[x]` nguyên và `b` nguyên: `b > 0.5a + 7` ⇔ `2b > a + 14` ⇔ `b >= floor((a + 14)/2) + 1 = a//2 + 8` (vì `floor((a + 14) / 2) = a//2 + 7`). Vậy `b ∈ [a//2 + 8, a]`. Khoảng này rỗng khi `a//2 + 8 > a` ⇔ `a <= 14`; không rỗng khi `a >= 15`, và khi đó `a` thuộc khoảng (vì `a//2 + 8 <= a`).

**Khẳng định 3 (đếm đúng).** Với mỗi người `x` tuổi `a >= 15`, số người `y` (kể cả `x`) có tuổi trong khoảng là `pre[a] - pre[a//2 + 7]` (tổng `cnt[b]` với `b ∈ [a//2 + 8, a]`). Vì `x` nằm trong khoảng, số người nhận **khác `x`** là số đó trừ `1`. Có `cnt[a]` người tuổi `a`, mọi người đều có cùng khoảng nên tổng đóng góp `cnt[a]·(pre[a] - pre[a//2 + 7] - 1)`. Người tuổi `a <= 14` không gửi được ai (khoảng rỗng). Cộng theo mọi tuổi ra đáp án; mỗi lời mời `(x, y)` được đếm đúng một lần, tại người gửi `x`.

**Cách 2.** Đóng góp của cặp tuổi `(a, b)` hợp lệ là `cnt[a]·cnt[b]` cặp có thứ tự (người tuổi `a`, người tuổi `b`); khi `a = b` có `cnt[a]·cnt[a]` cặp có thứ tự trong đó `cnt[a]` cặp là `(x, x)`, bỏ chúng cho `cnt[a]·(cnt[a] - 1)`.

**Cách 4 và 5.** Danh sách đã sắp xếp: số phần tử tuổi trong `[l, r]` là `bisect_right(r) - bisect_left(l)`. Hai con trỏ cho cùng hai chỉ số vì `a` không giảm nên các ngưỡng `0.5a + 7` và `a` không giảm, và chỉ số đầu tiên vượt ngưỡng chỉ tăng.

## 11. Time / Space Complexity

- **Cách 1 (mọi cặp):** Time `O(n²)`; Space `O(1)`.
- **Cách 2 (cặp tuổi):** Time `O(n + A²)` với `A = 120`; Space `O(A)`.
- **Cách 3 (tiền tố, nộp):** Time `O(n + A)`; Space `O(A)`.
- **Cách 4 (sắp xếp + bisect):** Time `O(n log n)`; Space `O(n)`.
- **Cách 5 (sắp xếp + hai con trỏ):** Time `O(n log n)` (sắp xếp) + `O(n)`; Space `O(n)`.

Hãy đo trên `n = 2·10^4` (cỡ tối đa) với tuổi ngẫu nhiên, và trường hợp xấu cho đếm trực tiếp (mọi người cùng một tuổi).

In [24]:
rng = random.Random(3)
N = 2 * 10 ** 4
cases = {
    "ngẫu nhiên 1..120": [rng.randint(1, 120) for _ in range(N)],
    "cùng một tuổi (50)": [50] * N,
    "ngẫu nhiên 15..30": [rng.randint(15, 30) for _ in range(N)],
}
methods = [
    ("age-pairs", count_by_age_pairs),
    ("prefix", count_prefix),
    ("bisect", count_bisect),
    ("2-ptr", count_two_pointers),
]
for name, ages in cases.items():
    results = []
    for mname, fn in methods:
        res, dt = timeit_once(fn, ages)
        results.append((mname, res, dt))
    assert len({r[1] for r in results}) == 1
    print(f"{name:22s} đáp án = {results[0][1]:>11d} | " + " | ".join(f"{m}: {dt * 1000:6.1f} ms" for m, _, dt in results))
assert count_prefix([50] * N) == N * (N - 1)
print("mọi người cùng tuổi 50: đáp án = n(n-1) =", N * (N - 1))

ngẫu nhiên 1..120      đáp án =    79385932 | age-pairs:    2.3 ms | prefix:    0.5 ms | bisect:    8.6 ms | 2-ptr:    6.3 ms
cùng một tuổi (50)     đáp án =   399980000 | age-pairs:    0.7 ms | prefix:    0.7 ms | bisect:    5.7 ms | 2-ptr:    3.0 ms
ngẫu nhiên 15..30      đáp án =   112247953 | age-pairs:    0.7 ms | prefix:    0.7 ms | bisect:    7.5 ms | 2-ptr:    5.6 ms
mọi người cùng tuổi 50: đáp án = n(n-1) = 399980000


Bốn cách nhanh đều chạy trong vài mili-giây đến vài chục mili-giây; chúng cho cùng đáp án. Cách vét cạn `O(n²)` ở `n = 2·10^4` cần cỡ `4·10^8` thao tác nên bị loại.

## 12. Bộ test

Bốn lớp test: ví dụ đề, edge case, phản ví dụ cho các cách "gần đúng", và đối chiếu có hệ thống/ngẫu nhiên (có seed) với oracle `count_brute` (đúng ba điều kiện của đề trên từng cặp người).

In [25]:
all_methods = {
    "age-pairs": count_by_age_pairs,
    "prefix": count_prefix,
    "bisect": count_bisect,
    "2-ptr": count_two_pointers,
    "Sol2": lambda a: Solution2().numFriendRequests(a),
    "Sol3": lambda a: Solution3().numFriendRequests(a),
    "Sol4": lambda a: Solution4().numFriendRequests(a),
    "Sol5": lambda a: Solution5().numFriendRequests(a),
    "submit": lambda a: Solution().numFriendRequests(a),
}

def run_all(ages, expected=None):
    exp = count_brute(ages) if expected is None else expected
    for name, fn in all_methods.items():
        got = fn(list(ages))
        assert got == exp, (name, ages, got, exp)
    return exp

for ages, expected in examples:
    assert run_all(ages, expected) == expected
print("ví dụ đề: OK")

ví dụ đề: OK


In [26]:
edge = [
    ([1], 0),                              # một người
    ([120], 0),
    ([14, 14], 0),                         # tuổi 14: khoảng rỗng (0.5*14+7 = 14, cần > 14)
    ([15, 15], 2),                         # tuổi 15: vừa đủ (0.5*15+7 = 14.5 < 15)
    ([15, 14], 0),                         # 14 <= 14.5 nên 15 không gửi cho 14
    ([16, 15], 1),                         # 16 gửi 15 (15 > 15?) -> 0.5*16+7 = 15, 15 <= 15 nên KHÔNG gửi
    ([17, 16], 1),                         # 0.5*17+7 = 15.5 < 16: 17 gửi cho 16
    ([100, 101], 0 + 1 - 1),               # 101 gửi 100? 0.5*101+7 = 57.5 < 100 <= 101: có; 100 gửi 101? 101 > 100: không
    ([99, 101], 0),                        # (ghi chú: cần đối chiếu bằng oracle ở dưới)
    ([50, 50, 50], 6),                     # ba người cùng tuổi: 3 * 2
    ([120, 120, 120, 120], 12),
    ([14, 15, 16, 17, 18, 19, 20], None),
]
for ages, expected in edge:
    got = run_all(ages)
    print(ages, "->", got)

[1] -> 0
[120] -> 0
[14, 14] -> 0
[15, 15] -> 2
[15, 14] -> 0
[16, 15] -> 0
[17, 16] -> 1
[100, 101] -> 1
[99, 101] -> 1
[50, 50, 50] -> 6
[120, 120, 120, 120] -> 12
[14, 15, 16, 17, 18, 19, 20] -> 6


Hai dòng dự đoán trong danh sách trên có kèm chú thích "(cần đối chiếu bằng oracle)", vì hai ca đó dễ nhầm: ví dụ `[16, 15]`: `0.5·16 + 7 = 15` và `15 <= 15` nên **không** gửi (kết quả đúng là `0`, không phải `1`), còn `[100, 101]` thì chỉ `101 -> 100` hợp lệ (kết quả `1`). Ta chốt lại bảng kỳ vọng bằng các `assert` chính xác dưới đây, mọi cách đều phải khớp.

In [27]:
checked = [
    ([1], 0), ([120], 0), ([14, 14], 0), ([15, 15], 2), ([15, 14], 0),
    ([16, 15], 0),          # 15 <= 0.5*16 + 7 = 15  ->  không gửi
    ([17, 16], 1),          # 16 > 15.5 và 16 <= 17  ->  17 gửi cho 16
    ([100, 101], 1),        # chỉ 101 -> 100
    ([99, 101], 1),         # 101 -> 99: 99 > 0.5*101+7 = 57.5, 99 <= 101, và (99 > 100? không) -> gửi
    ([50, 50, 50], 6), ([120, 120, 120, 120], 12),
    ([14, 15, 16, 17, 18, 19, 20], count_brute([14, 15, 16, 17, 18, 19, 20])),
]
for ages, expected in checked:
    assert run_all(ages, expected) == expected == count_brute(ages)
print("edge case: tất cả", len(checked), "trường hợp đúng (đã chốt bằng oracle)")

edge case: tất cả 12 trường hợp đúng (đã chốt bằng oracle)


Phản ví dụ cho các cách "gần đúng" mà người mới hay nghĩ ra:

- *Không trừ chính mình*: với `[16, 16]` đúng là `2` nhưng bản sai cho `4`.
- *Dùng `cnt[a]·cnt[a]` cho cặp cùng tuổi*: sai tương tự.
- *Biên sai `<` thay vì `<=`* ở điều kiện `0.5a + 7`: tuổi `16` bị gửi cho tuổi `15` oan (vì `15 = 0.5·16 + 7`).
- *Đếm `y` có tuổi lớn hơn* (sai chiều so sánh).

In [28]:
def wrong_no_self_exclusion(ages):
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    pre = list(accumulate(cnt))
    return sum(cnt[a] * (pre[a] - pre[a // 2 + 7]) for a in range(15, 121) if cnt[a])      # BUG: quên - 1

def wrong_strict_boundary(ages):
    n = len(ages)
    return sum(1 for i in range(n) for j in range(n)
               if i != j and ages[j] < 0.5 * ages[i] + 7 is False or (i != j and 0.5 * ages[i] + 7 < ages[j] <= ages[i]) or (i != j and ages[j] == 0.5 * ages[i] + 7))

def wrong_equal_pairs_squared(ages):
    cnt = Counter(ages)
    ans = 0
    for a, ca in cnt.items():
        for b, cb in cnt.items():
            if can_send(a, b):
                ans += ca * cb                                                                # BUG: cùng tuổi phải trừ ca
    return ans

print("[16, 16]       đúng =", count_brute([16, 16]), "| quên trừ chính mình =", wrong_no_self_exclusion([16, 16]), "| cnt^2 =", wrong_equal_pairs_squared([16, 16]))
print("[16, 15]       đúng =", count_brute([16, 15]), "| biên sai (bao cả ==) =", wrong_strict_boundary([16, 15]))
assert wrong_no_self_exclusion([16, 16]) != count_brute([16, 16])
assert wrong_equal_pairs_squared([16, 16]) != count_brute([16, 16])
assert wrong_strict_boundary([16, 15]) != count_brute([16, 15])

[16, 16]       đúng = 2 | quên trừ chính mình = 4 | cnt^2 = 4
[16, 15]       đúng = 0 | biên sai (bao cả ==) = 1


In [29]:
# Đối chiếu có hệ thống: MỌI dãy độ dài 1..4 trên tập tuổi "nhạy cảm" (các biên 14/15/16, 100/101, 120, ...)
from itertools import product

ages_set = [1, 14, 15, 16, 17, 30, 31, 99, 100, 101, 120]
total = 0
for length in range(1, 5):
    for tup in product(ages_set, repeat=length):
        run_all(list(tup))
        total += 1
print("vét cạn", total, "dãy tuổi (độ dài 1..4 trên 11 giá trị biên): mọi cách khớp oracle")

vét cạn 16104 dãy tuổi (độ dài 1..4 trên 11 giá trị biên): mọi cách khớp oracle


In [30]:
# Ngẫu nhiên có seed, đúng ràng buộc (1 <= ages <= 120)
rng = random.Random(2030)
for t in range(600):
    n = rng.randint(1, 60)
    mode = rng.choice(["uniform", "few_values", "teens", "old"])
    if mode == "uniform":
        ages = [rng.randint(1, 120) for _ in range(n)]
    elif mode == "few_values":
        pool = [rng.randint(1, 120) for _ in range(3)]
        ages = [rng.choice(pool) for _ in range(n)]
    elif mode == "teens":
        ages = [rng.randint(10, 30) for _ in range(n)]
    else:
        ages = [rng.randint(90, 120) for _ in range(n)]
    run_all(ages)
print("600 ca ngẫu nhiên: OK")

# Lớn: không dùng oracle O(n^2); đối chiếu các cách nhanh với nhau và với công thức đã biết
N = 2 * 10 ** 4
big_cases = [
    [rng.randint(1, 120) for _ in range(N)],
    [rng.randint(15, 25) for _ in range(N)],
    [120] * N,
    [14] * N,
    list(range(1, 121)) * (N // 120),
]
for ages in big_cases:
    base = count_prefix(ages)
    for name in ("age-pairs", "bisect", "2-ptr", "Sol2", "Sol4", "Sol5", "submit"):
        assert all_methods[name](ages) == base, name
assert count_prefix([120] * N) == N * (N - 1)
assert count_prefix([14] * N) == 0
print("test n = 2*10^4: OK")

600 ca ngẫu nhiên: OK
test n = 2*10^4: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ chính mình

Khoảng tuổi của `x` chứa cả chính `x` (khi `age[x] >= 15`). Đếm "số người trong khoảng" mà không trừ `1` sẽ cộng thêm một lời mời "gửi cho chính mình" cho mỗi người.

In [31]:
for ages in ([16, 16], [20, 30, 100, 110, 120], [16, 16, 16, 17]):
    print(ages, "| quên trừ chính mình:", wrong_no_self_exclusion(ages), "| đúng:", Solution().numFriendRequests(ages))
assert wrong_no_self_exclusion([20, 30, 100, 110, 120]) != 3

[16, 16] | quên trừ chính mình: 4 | đúng: 2
[20, 30, 100, 110, 120] | quên trừ chính mình: 8 | đúng: 3
[16, 16, 16, 17] | quên trừ chính mình: 13 | đúng: 9


### Lỗi 2 — Cặp cùng tuổi: `cnt²` thay vì `cnt·(cnt - 1)`

Khi duyệt cặp tuổi `(a, b)` mà `a == b`, người tuổi `a` gửi cho `cnt[a] - 1` người cùng tuổi (trừ chính mình), không phải `cnt[a]`. Quên điều này cho kết quả dư `cnt[a]`.

In [32]:
for ages in ([16, 16], [50, 50, 50], [18, 18, 17]):
    print(ages, "| cnt^2:", wrong_equal_pairs_squared(ages), "| đúng:", Solution().numFriendRequests(ages))
assert wrong_equal_pairs_squared([50, 50, 50]) == 9 and Solution().numFriendRequests([50, 50, 50]) == 6

[16, 16] | cnt^2: 4 | đúng: 2
[50, 50, 50] | cnt^2: 9 | đúng: 6
[18, 18, 17] | cnt^2: 7 | đúng: 4


### Lỗi 3 — Sai biên `<=` / `<` ở điều kiện `0.5·age + 7`

Điều kiện "không gửi" là `age[y] <= 0.5·age[x] + 7` (có dấu **bằng**). Nếu bạn viết `age[y] < 0.5·age[x] + 7` để loại, thì với `age[x] = 16`, `age[y] = 15` (`15 = 0.5·16 + 7`) lời mời bị cho phép oan. Tương tự biên `age <= 14`: `0.5·14 + 7 = 14` nên tuổi 14 không gửi cho cả chính tuổi 14.

In [33]:
def buggy_strict_less(ages):
    n = len(ages)
    return sum(1 for i in range(n) for j in range(n)
               if i != j and not (ages[j] < 0.5 * ages[i] + 7 or ages[j] > ages[i]))     # BUG: < thay vì <=

for ages in ([16, 15], [14, 14], [20, 17]):
    print(ages, "| sai (<):", buggy_strict_less(ages), "| đúng:", Solution().numFriendRequests(ages))
assert buggy_strict_less([16, 15]) == 1 and Solution().numFriendRequests([16, 15]) == 0
assert buggy_strict_less([14, 14]) == 2 and Solution().numFriendRequests([14, 14]) == 0

[16, 15] | sai (<): 1 | đúng: 0
[14, 14] | sai (<): 2 | đúng: 0
[20, 17] | sai (<): 1 | đúng: 0


### Lỗi 4 — Đếm sai chiều so sánh tuổi

Người **lớn** gửi cho người **nhỏ hoặc bằng** tuổi, không phải ngược lại. Nếu đếm `y` có `age[y] >= age[x]` thì chiều bị đảo. Ở ví dụ 2 (`[16,17,18]`) lời mời là `17 -> 16` và `18 -> 17`.

In [34]:
def buggy_reversed(ages):
    n = len(ages)
    return sum(1 for i in range(n) for j in range(n)
               if i != j and ages[i] <= ages[j] and ages[i] > 0.5 * ages[j] + 7)         # BUG: đảo vai trò x, y

for ages in ([16, 17, 18], [20, 30, 100, 110, 120]):
    print(ages, "| chiều đảo:", buggy_reversed(ages), "| đúng:", Solution().numFriendRequests(ages))
assert buggy_reversed([16, 17, 18]) == 2        # trùng hợp ở ví dụ này (hai chiều đối xứng về số lượng)
print("lưu ý: với ví dụ 2 số lượng trùng hợp bằng nhau, nên hãy thử nhiều ví dụ (kể cả cùng tuổi và tuổi 100+)")

[16, 17, 18] | chiều đảo: 2 | đúng: 2
[20, 30, 100, 110, 120] | chiều đảo: 3 | đúng: 3
lưu ý: với ví dụ 2 số lượng trùng hợp bằng nhau, nên hãy thử nhiều ví dụ (kể cả cùng tuổi và tuổi 100+)


### Lỗi 5 — Dùng `O(n²)` với `n = 2·10^4`

Cách vét cạn đúng nhưng cần `4·10^8` phép kiểm tra, quá thời gian. Ta đã đo ở mục 5: gấp đôi `n` thì thời gian gấp ~4. Hãy luôn tận dụng ràng buộc nhỏ của **giá trị** (`ages <= 120`) hoặc dùng sắp xếp.

In [35]:
rng = random.Random(4)
ages = [rng.randint(1, 120) for _ in range(3000)]
res, dt = timeit_once(count_brute, ages)
print(f"n = 3000: vét cạn mất {dt * 1000:.0f} ms; ngoại suy n = 20000 ~ {dt * (20000 / 3000) ** 2:.0f} giây")
res2, dt2 = timeit_once(count_prefix, [rng.randint(1, 120) for _ in range(20000)])
print(f"n = 20000: tổng tiền tố mất {dt2 * 1000:.1f} ms")

n = 3000: vét cạn mất 1320 ms; ngoại suy n = 20000 ~ 59 giây
n = 20000: tổng tiền tố mất 0.6 ms


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** đếm số cặp `(x, y)` thỏa điều kiện chỉ phụ thuộc vào **giá trị** nhỏ ⇒ **đếm theo giá trị** (counting) rồi nhân; điều kiện dạng "một khoảng giá trị" ⇒ **tổng tiền tố** hoặc **tìm kiếm nhị phân/hai con trỏ** trên dãy đã sắp xếp.
- Luôn **rút gọn điều kiện trước** (điều kiện thứ ba ở đây thừa); điều đó biến bài toán thành "đếm trong khoảng".
- Cặp **có thứ tự** `(x, y)` với `x != y`: khi `x` và `y` cùng giá trị nhớ trừ chính mình (`cnt·(cnt - 1)`).
- Chuyển ngưỡng thực sang số nguyên: `b > 0.5a + 7` ⇔ `2b > a + 14` ⇔ `b >= a//2 + 8`. Dùng số nguyên để tránh sai số và sai biên.
- Khi giá trị bị chặn nhỏ (`<= 120`), mảng đếm + tiền tố thắng sắp xếp; khi giá trị lớn, sắp xếp + `bisect`/hai con trỏ.
- Các bài cùng họ: **1347**, **1346** (đếm theo giá trị), **2605**, **1099. Two Sum Less Than K**, **167. Two Sum II - Input Array Is Sorted**, **611. Valid Triangle Number**, **1170. Compare Strings by Frequency of the Smallest Character**.

## 15. Biến thể

### 15.1. Mỗi người gửi bao nhiêu lời mời? (danh sách theo từng người)

Trả về `sent[i]` cho từng người `i`. Với tổng tiền tố, người tuổi `a` gửi `pre[a] - pre[a//2 + 7] - 1` lời mời (nếu `a >= 15`).

In [36]:
def requests_sent_each(ages):
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    pre = list(accumulate(cnt))
    return [pre[a] - pre[a // 2 + 7] - 1 if a >= 15 else 0 for a in ages]

def requests_sent_each_brute(ages):
    n = len(ages)
    return [sum(1 for j in range(n) if i != j and can_send(ages[i], ages[j])) for i in range(n)]

ages = [20, 30, 100, 110, 120]
print(requests_sent_each(ages), "| tổng =", sum(requests_sent_each(ages)))
rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(1, 120) for _ in range(rng.randint(1, 30))]
    assert requests_sent_each(a) == requests_sent_each_brute(a)
    assert sum(requests_sent_each(a)) == count_prefix(a)
print("OK: tổng các phần tử = đáp án của bài")

[0, 0, 0, 1, 2] | tổng = 3
OK: tổng các phần tử = đáp án của bài


### 15.2. Mỗi người **nhận** bao nhiêu lời mời?

Người tuổi `b` nhận lời mời từ mọi người `x` có tuổi `a` thỏa `b ∈ [a//2 + 8, a]`, tức `a ∈ [b, 2(b - 7) - 1]`... Cách đơn giản và chắc chắn: tính bằng mảng đếm duyệt mọi tuổi `a`.

In [37]:
def requests_received_each(ages):
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    recv_by_age = [0] * 121
    for a in range(15, 121):
        if not cnt[a]:
            continue
        for b in range(a // 2 + 8, a + 1):
            recv_by_age[b] += cnt[a] - (1 if a == b else 0)      # người tuổi a gửi cho mỗi người tuổi b (trừ chính mình)
    return [recv_by_age[b] for b in ages]

def requests_received_each_brute(ages):
    n = len(ages)
    return [sum(1 for i in range(n) if i != j and can_send(ages[i], ages[j])) for j in range(n)]

print(requests_received_each([16, 17, 18]), "(18 nhận 0, 17 nhận 1 từ 18, 16 nhận 1 từ 17)")
rng = random.Random(16)
for _ in range(300):
    a = [rng.randint(1, 120) for _ in range(rng.randint(1, 30))]
    assert requests_received_each(a) == requests_received_each_brute(a)
    assert sum(requests_received_each(a)) == count_prefix(a)
print("OK")

[1, 1, 0] (18 nhận 0, 17 nhận 1 từ 18, 16 nhận 1 từ 17)
OK


### 15.3. Đếm các cặp **kết bạn hai chiều** (mỗi người gửi cho người kia)

`x -> y` và `y -> x` cùng xảy ra khi `0.5·age[y] + 7 < age[x] <= age[y]` và `0.5·age[x] + 7 < age[y] <= age[x]`. Hai bất đẳng thức `age[x] <= age[y]` và `age[y] <= age[x]` buộc **cùng tuổi**. Và cùng tuổi `a` gửi cho nhau khi `a >= 15`. Vậy số cặp (không thứ tự) kết bạn hai chiều là `Σ_{a >= 15} C(cnt[a], 2)`.

In [38]:
from math import comb

def mutual_pairs(ages):
    cnt = Counter(ages)
    return sum(comb(c, 2) for a, c in cnt.items() if a >= 15)

def mutual_pairs_brute(ages):
    n = len(ages)
    return sum(1 for i in range(n) for j in range(i + 1, n) if can_send(ages[i], ages[j]) and can_send(ages[j], ages[i]))

assert mutual_pairs([16, 16]) == 1
assert mutual_pairs([16, 17, 18]) == 0
rng = random.Random(17)
for _ in range(300):
    a = [rng.randint(10, 40) for _ in range(rng.randint(1, 30))]
    assert mutual_pairs(a) == mutual_pairs_brute(a)
print("hai người kết bạn hai chiều <=> cùng tuổi và tuổi >= 15: OK")

hai người kết bạn hai chiều <=> cùng tuổi và tuổi >= 15: OK


### 15.4. Tổng quát hóa ngưỡng và tuổi lớn (không dùng mảng đếm)

Nếu tuổi có thể tới `10^9`, mảng đếm 121 phần tử không dùng được; Cách 4 (bisect) và Cách 5 (hai con trỏ) vẫn đúng vì chỉ dựa vào so sánh. Ta cũng có thể tham số hóa luật `age[y] > k·age[x] + c`.

In [39]:
def count_bisect_general(ages, num=1, den=2, offset=7):
    """x gửi cho y khi  num/den * age[x] + offset < age[y] <= age[x]  (so sánh bằng số nguyên)."""
    s = sorted(ages)
    ans = 0
    for a in s:
        # age[y] > (num*a + den*offset)/den  <=>  den*age[y] > num*a + den*offset
        thr = (num * a + den * offset) // den          # age[y] > thr  <=>  age[y] >= thr + 1
        lo = bisect_left(s, thr + 1)
        hi = bisect_right(s, a)
        if hi > lo:
            ans += hi - lo - 1
    return ans

def count_brute_general(ages, num=1, den=2, offset=7):
    n = len(ages)
    return sum(1 for i in range(n) for j in range(n)
               if i != j and den * ages[j] > num * ages[i] + den * offset and ages[j] <= ages[i])

rng = random.Random(18)
for _ in range(300):
    a = [rng.randint(1, 10 ** 9) for _ in range(rng.randint(1, 25))]
    assert count_bisect_general(a) == count_brute_general(a)
    a = [rng.randint(1, 120) for _ in range(rng.randint(1, 25))]
    num, den, off = rng.choice([(1, 2, 7), (1, 3, 5), (2, 3, 0), (3, 4, 2)])
    assert count_bisect_general(a, num, den, off) == count_brute_general(a, num, den, off)
assert count_bisect_general([20, 30, 100, 110, 120]) == 3
print("tổng quát hóa: OK (kể cả tuổi tới 10^9 và các tham số khác)")

tổng quát hóa: OK (kể cả tuổi tới 10^9 và các tham số khác)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp người | Kiểm tra ba điều kiện từng cặp | `O(n²)` | `O(1)` | Chỉ làm oracle, `n` nhỏ |
| 2. Cặp tuổi | Đếm theo tuổi, nhân `cnt[a]·cnt[b]` | `O(n + A²)` | `O(A)` | Giá trị bị chặn nhỏ (`A = 120`) |
| 3. Tổng tiền tố (nộp) | Khoảng `[a//2 + 8, a]` đếm bằng `pre` | `O(n + A)` | `O(A)` | Mặc định với đề này |
| 4. Sắp xếp + bisect | Hai lần `bisect` cho khoảng | `O(n log n)` | `O(n)` | Giá trị lớn, không dùng mảng đếm được |
| 5. Sắp xếp + hai con trỏ | Hai mốc chỉ tiến lên | `O(n log n)` (sắp xếp) | `O(n)` | Giá trị lớn; khi đã sắp xếp thì `O(n)` |

Cách 3 nhanh nhất khi tuổi nhỏ; cách 4 và 5 là lời giải tổng quát.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(16 // 2 + 8)
print(bisect_right([1, 3, 3, 5], 3))
print(Counter([16, 16, 17])[16])
```

In [40]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `ages = [15, 15, 16, 17, 30]`: với mỗi tuổi có người, ghi khoảng tuổi nhận `[a//2 + 8, a]`, số lời mời mỗi người tuổi đó gửi, rồi tổng.

In [41]:
# Bài 2: điền kết quả tính tay
hand_sent_by_age = None     # dict {tuổi: số lời mời MỖI người tuổi đó gửi}
hand_answer = None          # tổng số lời mời

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 3 (tổng tiền tố).

In [42]:
def exercise_3_prefix(ages: List[int]) -> int:
    # TODO 1: tạo cnt = [0] * 121 và đếm tuổi
    # TODO 2: pre = list(accumulate(cnt))
    ans = 0
    # TODO 3: với a từ 15 đến 120 mà cnt[a] > 0:
    #     TODO 4: ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)
    return None


# assert exercise_3_prefix([16, 16]) == 2
# assert exercise_3_prefix([16, 17, 18]) == 2
# assert exercise_3_prefix([20, 30, 100, 110, 120]) == 3

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_two_pointers(ages)`: sắp xếp, rồi dùng hai con trỏ `lo` và `hi` chỉ tiến lên (dùng số nguyên `2*s[lo] <= a + 14`).

In [43]:
def exercise_4_two_pointers(ages: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_two_pointers([16, 16]) == 2
# assert exercise_4_two_pointers([16, 17, 18]) == 2
# assert exercise_4_two_pointers([20, 30, 100, 110, 120]) == 3

### Bài 5 — Mini challenge: số cặp kết bạn hai chiều (khó)

Viết `exercise_5_mutual(ages)` trả về số **cặp không thứ tự** `{i, j}` mà **cả hai** cùng gửi lời mời cho nhau (`i` gửi `j` và `j` gửi `i`). Gợi ý: phân tích hai bất đẳng thức cùng lúc; xem biến thể 15.3.

```python
exercise_5_mutual([16, 16]) -> 1
exercise_5_mutual([16, 17, 18]) -> 0
exercise_5_mutual([14, 14, 15, 15, 15]) -> 3
```

In [44]:
def exercise_5_mutual(ages: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_mutual([16, 16]) == 1
# assert exercise_5_mutual([16, 17, 18]) == 0
# assert exercise_5_mutual([14, 14, 15, 15, 15]) == 3

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `//` là chia lấy phần nguyên; `bisect_right` trả về chỉ số sau các phần tử bằng giá trị; `Counter` đếm.
- Mức 2: `16 // 2 = 8`; trong `[1,3,3,5]` có hai số `3` ở chỉ số 1 và 2.
- Mức 3: `16`, `3`, `2`.

**Bài 2**
- Mức 1: tuổi 15: khoảng `[15, 15]`; tuổi 16: `[16, 16]`; tuổi 17: `[16, 17]`; tuổi 30: `[23, 30]`.
- Mức 2: số người trong khoảng (kể cả chính mình): 15 → 2 (hai người tuổi 15); 16 → 1; 17 → 2 (16 và 17); 30 → 1. Trừ chính mình.
- Mức 3: mỗi người tuổi 15 gửi `1`, tuổi 16 gửi `0`, tuổi 17 gửi `1`, tuổi 30 gửi `0`. Tổng `2·1 + 1·0 + 1·1 + 1·0 = 3`.

**Bài 3**
- Mức 1: giống mục 7.2.
- Mức 2: `cnt = [0] * 121; for a in ages: cnt[a] += 1`.
- Mức 3: `for a in range(15, 121): if cnt[a]: ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)`.

**Bài 4**
- Mức 1: `s = sorted(ages)`, `lo = hi = 0`.
- Mức 2: với mỗi `a` trong `s`: `while lo < n and 2*s[lo] <= a + 14: lo += 1` và `while hi < n and s[hi] <= a: hi += 1`.
- Mức 3: nếu `hi > lo` thì `ans += hi - lo - 1`.

**Bài 5**
- Mức 1: hai điều kiện `x -> y` và `y -> x` cùng đúng cần `age[x] <= age[y]` và `age[y] <= age[x]`.
- Mức 2: vậy hai người phải **cùng tuổi**; khi đó cần tuổi `a >= 15` (khoảng không rỗng).
- Mức 3: `sum(comb(cnt[a], 2) for a >= 15)`. Với `[14,14,15,15,15]`: tuổi 14 không tính, tuổi 15 có `C(3, 2) = 3`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [45]:
ans1 = [16 // 2 + 8, bisect_right([1, 3, 3, 5], 3), Counter([16, 16, 17])[16]]
print(ans1)
assert ans1 == [16, 3, 2]

[16, 3, 2]


### Đáp án Bài 2

In [46]:
ans2_sent = {15: 1, 16: 0, 17: 1, 30: 0}
ans2_answer = 3
ages = [15, 15, 16, 17, 30]
sent = requests_sent_each(ages)
for a, expected in ans2_sent.items():
    assert sent[ages.index(a)] == expected
assert sum(sent) == ans2_answer == count_brute(ages) == Solution().numFriendRequests(ages)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [47]:
def ans3_prefix(ages: List[int]) -> int:
    cnt = [0] * 121
    for a in ages:
        cnt[a] += 1
    pre = list(accumulate(cnt))
    ans = 0
    for a in range(15, 121):
        if cnt[a]:
            ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)
    return ans

assert ans3_prefix([16, 16]) == 2
assert ans3_prefix([16, 17, 18]) == 2
assert ans3_prefix([20, 30, 100, 110, 120]) == 3
rng = random.Random(51)
for _ in range(300):
    a = [rng.randint(1, 120) for _ in range(rng.randint(1, 40))]
    assert ans3_prefix(a) == count_brute(a)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [48]:
def ans4_two_pointers(ages: List[int]) -> int:
    s = sorted(ages)
    n = len(s)
    lo = hi = 0
    ans = 0
    for a in s:
        while lo < n and 2 * s[lo] <= a + 14:
            lo += 1
        while hi < n and s[hi] <= a:
            hi += 1
        if hi > lo:
            ans += hi - lo - 1
    return ans

assert ans4_two_pointers([16, 16]) == 2
assert ans4_two_pointers([16, 17, 18]) == 2
assert ans4_two_pointers([20, 30, 100, 110, 120]) == 3
rng = random.Random(52)
for _ in range(300):
    a = [rng.randint(1, 120) for _ in range(rng.randint(1, 40))]
    assert ans4_two_pointers(a) == count_brute(a)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [49]:
def ans5_mutual(ages: List[int]) -> int:
    cnt = Counter(ages)
    return sum(comb(c, 2) for a, c in cnt.items() if a >= 15)

assert ans5_mutual([16, 16]) == 1
assert ans5_mutual([16, 17, 18]) == 0
assert ans5_mutual([14, 14, 15, 15, 15]) == 3
rng = random.Random(53)
for _ in range(300):
    a = [rng.randint(10, 40) for _ in range(rng.randint(1, 30))]
    assert ans5_mutual(a) == mutual_pairs_brute(a)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def numFriendRequests(self, ages: List[int]) -> int:
        cnt = [0] * 121                     # cnt[a] = số người tuổi a
        for a in ages:
            cnt[a] += 1
        pre = list(accumulate(cnt))         # pre[i] = số người tuổi <= i
        ans = 0
        for a in range(15, 121):            # tuổi <= 14 không gửi được ai
            if cnt[a]:
                # người nhận: tuổi trong [a//2 + 8, a]; trừ chính mình
                ans += cnt[a] * (pre[a] - pre[a // 2 + 7] - 1)
        return ans
```

Nhớ ba ý: **điều kiện thực chất là `0.5a + 7 < b <= a` (điều kiện thứ ba thừa)**, **khoảng nguyên là `[a//2 + 8, a]`**, **nhớ trừ chính mình khi đếm**.

### Checklist trước khi submit

- [ ] Đã rút gọn điều kiện thành `0.5a + 7 < b <= a` và đổi sang số nguyên `b >= a//2 + 8`.
- [ ] Tuổi `<= 14` không gửi được cho ai (khoảng rỗng).
- [ ] Đã trừ chính mình (hoặc dùng `cnt·(cnt - 1)` cho cặp cùng tuổi).
- [ ] Tính **cặp có thứ tự**: `x -> y` và `y -> x` là hai lời mời khác nhau.
- [ ] Biên `<=` / `<` đúng; đã thử `[14,14]`, `[15,15]`, `[16,15]`.
- [ ] Đã thử `n = 1`, mọi người cùng tuổi, tuổi `>= 100`.
- [ ] Time `O(n + 120)`, Space `O(121)`.